#### Filtramos las películas con año de lanzamiento mayores o iguales al 2015

In [0]:
movie_df = spark.read.table("movie_silver.movies").filter("year_release_date >= 2015")

In [0]:
production_country_df = spark.read.table("movie_silver.productions_countries")

In [0]:
country_df = spark.read.table("movie_silver.countries")

#### Join "production_countries" y "countries"

In [0]:
countries_prod_count_df = production_country_df.join(country_df,
                                    production_country_df.country_id == country_df.country_id,
                                    "inner") \
                                    .select(country_df.country_name, production_country_df.movie_id)

#### Join "movies" y "countries_prod_count_df"

In [0]:
results_group_movie_country = movie_df.join(countries_prod_count_df,
                                    movie_df.movie_id == countries_prod_count_df.movie_id,
                                    "inner") \
                                    .select(movie_df.year_release_date, movie_df.budget, 
                                            movie_df.revenue, countries_prod_count_df.country_name)

#### 1-Calcular el total de presupuesto y total de ingresos de las peliculas agrupadas por el año de lanzamiento y el genero
#### 2-Hacer un ranking particionado por el año de lanzamiento ordenado por el total del presupuesto y el total de ingresos de forma descendente

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, rank, desc

In [0]:
df_final = results_group_movie_country.groupBy("year_release_date", "country_name") \
        .agg(
            sum("budget").alias("total_budget"),
            sum("revenue").alias("total_revenue")
        ) \
        .withColumn("rank", rank().over(Window.partitionBy("year_release_date").orderBy(desc("total_budget"),desc("total_revenue"))))

##### Escribir datos en una managed table en el contenedor gold

In [0]:
df_final.write.mode("overwrite").format("delta").saveAsTable("movie_gold.results_group_movie_country")

In [0]:
%sql
SELECT * FROM movie_gold.results_group_movie_country